# Large Run Step 6: Additional Plotting Driver

Purpose: create optional diagnostic figures from existing compact tables and bounded samples, without loading full QC or metric inventories in the notebook.

Outputs: waveform comparison figures, station/event maps, flexible metric plots, and selected diagnostics.


## Setup
Purpose: load config/output paths and shared settings.

Outputs: printed paths and helper variables.


In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
repo_root = runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from pathlib import Path
import os
import pandas as pd
from IPython.display import display

from spatial_vtk.config import (
    notebook_run_context,
    notebook_figure_sidecar_settings,
    prepare_notebook_geospatial_environment,
    print_notebook_context,
)
from spatial_vtk.io import (
    load_output_table,
    output_group_namespace,
    output_group_status_frame,
    output_status_frame,
    preview_output_table,
    preview_table,
)

prepare_notebook_geospatial_environment()
context = notebook_run_context(run_scenario=os.environ.get("SVTK_RUN_SCENARIO", "tutorial"))
cfg = context.cfg
repo_root = context.repo_root
config_path = context.config_path
outputs_root = context.outputs_root
tables_dir = context.tables_dir
figures_dir = context.figures_dir
slurm_dir = context.slurm_dir
logs_dir = context.logs_dir
RUN_LOCAL = context.run_local
SUBMIT_SLURM = context.submit_slurm
OVERWRITE = context.overwrite
QC_CHUNKSIZE = context.qc_chunksize
PREVIEW_ROWS = context.preview_rows

print_notebook_context(context)


## Resolve Plotting Inputs
Purpose: verify that comparison-eligible records and metric outputs are ready.

Outputs: status table only.


In [ ]:
step_outputs = output_group_namespace("step_06_plotting")
comparison_eligible_path = step_outputs.comparison_eligible_path
event_station_path = step_outputs.event_station_path
metrics_long_path = step_outputs.metrics_long_path
metrics_enriched_path = step_outputs.metrics_enriched_path
waveform_fig_path = step_outputs.event_trace_comparison_path

display(output_group_status_frame("step_06_plotting"))


## Render Bounded Waveform Comparison
Purpose: load only a small comparison-eligible sample and render one waveform diagnostic figure.

Outputs: `event_trace_comparison` figure when `SVTK_MAKE_FIGURES=1`.


In [ ]:
MAKE_FIGURES = os.environ.get("SVTK_MAKE_FIGURES", "0") == "1"
WAVEFORM_FIGURE_SIDECARS = notebook_figure_sidecar_settings("waveform", figure_dir=figures_dir)
if not MAKE_FIGURES:
    print("Skipping figures. Set SVTK_MAKE_FIGURES=1 to render them.")
elif not all(path.exists() for path in [comparison_eligible_path, event_station_path]):
    print("Comparison-eligible records or event-station records are not ready yet.")
else:
    from spatial_vtk.qc import build_qc_waveform_comparison_records, load_comparison_eligible_records
    from spatial_vtk.visualize.waveforms import plot_event_trace_comparison
    event_stations = load_output_table("event_station_records")
    sample = load_comparison_eligible_records(comparison_eligible_path, max_records=12, chunksize=QC_CHUNKSIZE)
    records = build_qc_waveform_comparison_records(event_stations, comparison_eligible=sample, max_records=12)
    plot_event_trace_comparison(records, showfig=False, savefig=True, **WAVEFORM_FIGURE_SIDECARS.kwargs())
    print(f"Wrote {waveform_fig_path}")


## Preview Metric Plot Inputs
Purpose: inspect only the first rows of the metric table selected for plotting.

Outputs: bounded preview table.


In [ ]:
metric_source = metrics_enriched_path if metrics_enriched_path.exists() else metrics_long_path
if metric_source.exists():
    display(preview_table(metric_source, nrows=PREVIEW_ROWS))
else:
    print("Metric plotting source is not ready yet.")


## Region Boxplot with Comparison Table
Purpose: reproduce the tutorial region-comparison boxplot with its bootstrap comparison table using a bounded metric sample.


In [ ]:
MAKE_FIGURES = os.environ.get("SVTK_MAKE_FIGURES", "0") == "1"
if not MAKE_FIGURES:
    print("Skipping region boxplot. Set SVTK_MAKE_FIGURES=1 to render it.")
else:
    from spatial_vtk.spatial.plot import write_large_run_region_boxplot

    metric_source = metrics_enriched_path if metrics_enriched_path.exists() else metrics_long_path
    REGION_FIGURE_ROWS = int(os.environ.get("SVTK_REGION_FIGURE_ROWS", "200000"))
    REGION_BOX_METRIC = os.environ.get("SVTK_REGION_BOX_METRIC", "PGA")
    REGION_BOX_PASSBAND = os.environ.get("SVTK_REGION_BOX_PASSBAND", "2-3 sec")
    REGION_BOX_COMPONENT = os.environ.get("SVTK_REGION_BOX_COMPONENT") or None
    REGION_BOX_MODEL = os.environ.get("SVTK_REGION_BOX_MODEL") or None
    REGION_VALUE_COL = os.environ.get("SVTK_REGION_VALUE_COL", "log2_residual")
    REGION_COMPARE_TO = os.environ.get("SVTK_REGION_COMPARE_TO", "LA Basin")
    REGION_SHOWFIG = os.environ.get("SVTK_FIGURE_SHOWFIG", "0") == "1"
    region_sidecars = notebook_figure_sidecar_settings("region", figure_dir=figures_dir / "metrics", default_rows=1000)

    result = write_large_run_region_boxplot(
        metric_source,
        figure_dir=figures_dir / "metrics",
        metric=REGION_BOX_METRIC,
        passband=REGION_BOX_PASSBAND,
        component=REGION_BOX_COMPONENT,
        model=REGION_BOX_MODEL,
        value_col=REGION_VALUE_COL,
        compare_to=REGION_COMPARE_TO,
        max_rows=REGION_FIGURE_ROWS,
        output_prefix="additional_region_boxplot",
        **region_sidecars.kwargs(),
        annotate_if_missing=False,
        overwrite=OVERWRITE,
        showfig=REGION_SHOWFIG,
    )
    print(result.message)
